# ENG-209, Partie 2, Série 9 (Cours)

## Introduction à la descente de gradient

In [ ]:
import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
from numpy.typing import NDArray

In [ ]:
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size']      = 10

In [ ]:
data: pd.DataFrame = pd.read_csv("../data/heart/heartData.csv")

In [ ]:
data.head()

In [ ]:
fig, ax = plt.subplots()
ax.scatter(
    x=data.biking,
    y=data['heart.disease'],
    color='#1f77b4',
    marker='.',
    label=r"Mesures $(x,y)$",
)
ax.grid(True)
ax.legend(loc='best')
ax.set_title("Vélo au travail vs santé cardiaque\nMesures en % de la population dans chaque ville", fontsize=12)
ax.set_xlabel("Vélo au travail (% de la population)")
ax.set_ylabel("Maladies cardiaques (% de la population)")
ax.tick_params(axis='both', labelsize=10) 
plt.show()

## Descente de gradient

### Application à la régression linéaire

In [ ]:
N: int = len(data)

# Soyons explicite et utilisons les shapes 2D (N,1), pour être éviter les surprises d'algèbre linéaire erronées.
X: NDArray[np.float64] = data[['biking']].to_numpy()
y: NDArray[np.float64] = data[['heart.disease']].to_numpy()

In [ ]:
X_tilde: NDArray[np.float64] = np.column_stack([np.ones(N), X])

#### Solution "closed form" par la méthode des moindres carrés ordinaire (OLS)

**Rappel**: $\bf{w}^* = \left( \bf{\tilde{x}}^T \bf{\tilde{x}} \right)^{-1} \bf{\tilde{x}}^T y$

In [ ]:
w_star: NDArray[np.float64] = np.dot(np.dot(np.linalg.inv(np.dot(X_tilde.T,X_tilde)),X_tilde.T), y)

In [ ]:
print(f"(w_0, w_1) = ({w_star[0][0]:2.2f}, {w_star[1][0]:2.2f})")

#### Solution par descente de gradient (méthode itérative)

Initialisons $w$ initiale, le taux d'apprentissage $\alpha$ et mettons une limite sur le nombre d'itérations max.

Plus le taux d’apprentissage $\alpha$ est élevé, plus la convergence initiale est rapide, mais le processus peut alors osciller autour du minimum. Et attention à ne pas le confondre avec $\alpha$ de la régularisation !

In [ ]:
w      = np.array([[9],[0.6]])    # vecteur de poids initial, shape (N,1)
alpha  = 0.001                    # taux d'apprentissage (learning rate)
maxits = 50000                    # nombre maximal d'itérations


Prenons la fonction de coût quadratique (Squared Loss) :

$$
g_{sl}(w) = \frac{1}{2N} \|y - \tilde{X}w\|_2^2
$$

Rappel, la norme euclidienne au carré ($\ell_2$) est définie par :

$$
\|u\|_2^2=\sum_i^N u_i^2
$$

In [ ]:
g_sl = lambda w: (1/(2*N))*(np.sum( (y - X_tilde.dot(w))**2) )

Le gradient de la fonction de coût $g_{sl}(w)$ est alors :

$$
\nabla g_{sl}(w) = - \frac{1}{N} \tilde{X}^\top (y - \tilde{X} w)
$$

In [ ]:
gradient_sl = lambda w: (-1/N)*X_tilde.T.dot(y - X_tilde.dot(w))

Calculons itérativement:

$$
\mathbf{w}^{\,k} = \mathbf{w}^{\,k-1} - \alpha \nabla g\left(\mathbf{w}^{k-1}\right)
$$

In [ ]:
for i in range(maxits):
    w = w - alpha*gradient_sl(w)

In [ ]:
print(f"(w_0, w_1) = ({w[0][0]:2.2f}, {w[1][0]:2.2f})")

Il a fallu 50000 itérations pour arriver à une solution proche de la solution OLS.

Pourquoi?

#### Surface de coût $\left( w_0, w_1, g_{sl}\right)$

In [ ]:
# Préparons les variables pour afficher cette surface
w_0: NDArray[np.float64] = np.arange(9, 25, 0.5, dtype=np.float64)
w_1: NDArray[np.float64] = np.arange(-1, 0.6, 0.05, dtype=np.float64)

W_0, W_1 = np.meshgrid(w_0, w_1)
w_grid = np.dstack((W_0, W_1))

# Coûts
g_sl_grid: NDArray[np.float64] = np.array([g_sl(w_it.reshape(2,1)) for w_it in w_grid.reshape(-1,2)])
cost_matrix = g_sl_grid.reshape(W_0.shape)

In [ ]:
import plotly.graph_objects as go

figx = go.Figure(data = [
    go.Surface(
        z=cost_matrix,
        x=w_0,
        y=w_1,
        opacity=0.8,
        contours={
            "z": dict(
                show=True,
                usecolormap=True,
                highlightcolor="limegreen",
                project_z=True,
            )
        }
    ),
])

figx.update_layout(scene = dict(
                    xaxis_title=r'w_0',
                    yaxis_title=r'w_1',
                    zaxis_title='Coût quadratique'),
                    width=800, height=800)

# select a different renderer if the figure doesn't appear, possible choices are:
# notebook, notebook_connected, browser, vscode
figx.show(renderer='browser')

Réponse: la descente de gradient est rapide lorsque $w_1$ est loin de sa valeur optimale, mais ralentit à l’approche du minimum, car la surface est très plate dans cette région.

#### Normalisation standard

Essayons de normaliser les variables dépendantes $\tilde{X} = \frac{\left( \tilde{X} - \mu_{\tilde{X}} \right)}{\sigma_{\tilde{X}}}$ et indépendante $y = \frac{\left( y - \mu_y \right)}{\sigma_y}$

In [ ]:
X = data[['biking']].to_numpy()
y = data[['heart.disease']].to_numpy()
X_norm = (X - X.mean())/(X.std())
X_tilde = np.column_stack([np.ones(N),  X_norm])

y = (y - y.mean())/(y.std())

#### Solution "closed form" des variables normalisées par la méthode des moindres carrés ordinaire (OLS)

In [ ]:
w_star = np.dot(np.dot(np.linalg.inv(np.dot(X_tilde.T,X_tilde)),X_tilde.T), y)

In [ ]:
print(f"(w_0, w_1) = ({w_star[0][0]:2.2f}, {w_star[1][0]:2.2f})")

In [ ]:
fig, ax = plt.subplots()

ax.scatter(X_norm, y, color='#1f77b4', marker='.', label=r"Mesures $(x,y)$")
ax.scatter(X_norm, X_tilde.dot(w_star), color='red', marker='.', label=r"Prédictions $(x,\hat{y})$")
ax.plot([X_norm.min(),X_norm.max()],
        [w_star[0][0]+w_star[1][0]*X_norm.min(),w_star[0][0]+w_star[1][0]*X_norm.max()],
        color='black',
        linestyle='--',
        label=rf"Modèle $y={w_star[0][0]:2.2f}{w_star[1][0]:+2.2f}x$"
)

ax.legend(loc='best')
ax.set_xlabel("Vélo au travail (% de la population)")
ax.set_ylabel("Maladies cardiaques (% de la population)")
plt.show()

#### Solution par descente de gradient (méthode itérative)

Calculons les valeurs successives de $w$ de façon itérative,
et conservons leur historique afin de visualiser la convergence.

In [ ]:
w      = np.array([[-2],[-3]], dtype=np.float64)
alpha  = 0.1
maxits = 200

In [ ]:
w_0_history = []
w_1_history = []
g_history   = []

for i in range(maxits):
    w_0_history.append(w[0][0])
    w_1_history.append(w[1][0])
    g_history.append(g_sl(w))
    
    w = w - alpha*gradient_sl(w)

In [ ]:
print(f"(w_0, w_1) = ({w[0][0]:2.2f}, {w[1][0]:2.2f})")

#### Surface de coût calculée à partir des variables normalisées

In [ ]:
# Préparons les variables pour afficher cette surface
w_0: NDArray[np.float64] = np.arange(-2, 2.05, 0.05, dtype=np.float64)
w_1: NDArray[np.float64] = np.arange(-3, 1.05, 0.05, dtype=np.float64)

W_0, W_1 = np.meshgrid(w_0, w_1)
w_grid = np.dstack((W_0, W_1))

# Coûts
g_sl_grid: NDArray[np.float64] = np.array([g_sl(w_it.reshape(2,1)) for w_it in w_grid.reshape(-1,2)])
cost_matrix = g_sl_grid.reshape(W_0.shape)

In [ ]:
import plotly.graph_objects as go

figx = go.Figure(data = [
    go.Surface(
        z=cost_matrix,
        x=w_0,
        y=w_1,
        opacity=0.8,
        contours={
            "z": dict(
                show=True,
                usecolormap=True,
                highlightcolor="limegreen",
                project_z=True,
            )
        }
    ),
    go.Scatter3d(
        x=w_0_history, y=w_1_history, z=g_history, mode ='markers', marker_size=4, marker_color='blue'
    )
])

figx.update_layout(scene = dict(
                    xaxis_title='w_0',
                    yaxis_title='w_1',
                    zaxis_title='Coût quadratique'),
                    width=800,height=800)

#figx.show(renderer='notebook')
figx.show(renderer='browser')

Nous vérifions que la fonction de coût est convexe, condition nécessaire pour garantir la convergence de la méthode.

#### Essayons avec une autre fonction de coût: somme des différences absolues

Prenons la fonction de coût des différences absolues (Absolute error loss) :

$$
g_{al}(w) = \frac{1}{N} \|y - \tilde{X}w\|_1
$$

Rappel, la norme $\ell_1$ est définie par :

$$
\|u\|_1=\sum_i^N |u_i|
$$

In [ ]:
g_al = lambda w: (np.sum( np.absolute(X_tilde.dot(w) - y ) ) / N )

Nous dénotons _sign_ le vecteur des signes, $-1$ ou $1$, de 
chaque composante de $\tilde{X}w - y$.

Le sous-gradient de la perte absolue s'écrit :

$$
\nabla g_{al}(w)
= \frac{1}{N}\,\tilde{X}^{\top}\,
\operatorname{sign}\!\big(\tilde{X}w - y\big).
$$


In [ ]:
gradient_al = lambda w: np.sum(X_tilde.T.dot( np.sign( X_tilde.dot(w) - y )) / N )

In [ ]:
w      = np.array([[-2],[-3]])
alpha  = 0.5
maxits = 1000

Nous calculons $w$ itérativement, en diminuant le pas $\alpha$ à chaque itération $i$, comme $\alpha / \sqrt{i+1}$.

Cela permet de ralentir la convergence à proximité de la discontinuité de la fonction de coût absolue moyen, près de la valeur optimale.

In [ ]:
w_0_history = []
w_1_history = []
g_history   = []

for i in range(maxits):
    w_0_history.append(w[0][0])
    w_1_history.append(w[1][0])
    g_history.append(g_al(w))
    
    w = w - (alpha/np.sqrt((i+1)))*gradient_al(w)

In [ ]:
print(f"(w_0, w_1) = ({w[0][0]:2.2f}, {w[1][0]:2.2f})")

In [ ]:
# Préparons les variables pour afficher cette surface

w_0 = np.arange(-2, 2, 0.02)
w_1 = np.arange(-3, 2, 0.02)

w_grid = np.array(np.meshgrid(w_0, w_1)).T.reshape(-1,2)

g_al_grid: NDArray[np.float64] = np.array([g_al(w_it.reshape(2,1)) for w_it in w_grid.reshape(-1,2)])

cost_matrix = np.array(g_al_grid).reshape(len(w_0), len(w_1))

In [ ]:
import plotly.graph_objects as go

figx = go.Figure(
    data = [
        go.Surface(
            x=w_0, y=w_1, z=cost_matrix, opacity=0.6,
            contours={"z": dict(
                show=True,
                usecolormap=True,
                highlightcolor="limegreen",
                project_z=True,
            )
            }
        ),
        go.Scatter3d(
            x=w_0_history, y=w_1_history, z=g_history,
            mode ='markers', marker_size=4, marker_color='blue')

    ]
)

figx.update_layout(scene = dict(
                    xaxis_title='w_0',
                    yaxis_title='w_1',
                    zaxis_title='Cost'),
                    width=800, height=800)

# figx.show(renderer='notebook')
figx.show(renderer='browser')

Nous vérifions que la fonction de coût absolue moyenne est convexe, condition nécessaire pour garantir la convergence de la méthode.

Notez que, au minimum, le gradient est théoriquement discontinu, ce qui crée un “tip” pointu. Nous ne le voyons pas clairement ici à cause de la discrétisation et du pas d’échantillonnage.

## Avec la bibliothèque Scikit Learn

In [ ]:
from sklearn.linear_model import SGDRegressor
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

In [ ]:
X = data[['biking']].to_numpy() # 2D
y = data['heart.disease'].to_numpy() #1D

In [ ]:
from sklearn.model_selection import train_test_split

x_train, x_test, y_train, y_test = train_test_split(X, y, test_size=0.2)

#### Descente de gradient stochastique

Dans la descente de gradient stochastique (ou « en ligne »), la vraie valeur du gradient de $g(\bf{w})$ est approchée par le gradient d'une seule composante de la somme (c-à-d. d'un seul exemple): Alors que l'algorithme parcourt l'ensemble d'apprentissage, il réalise cette étape de mise-à-jour pour chaque exemple.

Plusieurs parcours de l'ensemble d'apprentissage peuvent être nécessaires avant la convergence de la méthode. Si cela est nécessaire, les données sont habituellement mélangées à chaque parcours, afin d'éviter les cycles.

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn import metrics
from sklearn.pipeline import Pipeline

In [ ]:
# Pipeline, avec normalisation standard set Stochastic Gradient Descent (descente de gradient)
regressor: Pipeline = Pipeline(steps=[
        ('standardisation', StandardScaler()),
        ('regression', SGDRegressor(loss="squared_error", max_iter=1000, tol=1e-5)), # Régression itérative classique (pas de régularisation)
    ]
)

In [ ]:
regressor.fit(x_train, y_train)

In [ ]:
y_test_predict: NDArray[np.float64] = regressor.predict(x_test)

L’intercept et le coefficient sont calculés sur les données normalisées. Pour les exprimer sur les données d’origine, il faut appliquer la transformation inverse.

In [ ]:
# Note: les attributs coef_ et intercept_ peuvent être des ndarray ou des float64
# selon les dimensions de X et y. Dans notre exemple, nous avons un seul coefficient
# et un seul intercept, donc nous pouvons utiliser .item() pour les convertir en float64.
coef: np.float64 = (regressor.named_steps['regression'].coef_ / 
                    regressor.named_steps['standardisation'].scale_).item()
intercept: np.float64 = (regressor.named_steps['regression'].intercept_.item() - 
                         coef * regressor.named_steps['standardisation'].mean_.item())

In [ ]:
fig, ax = plt.subplots()


ax.scatter(x_train, y_train, color='#1f77b4', marker='.', label=r"Données $(x,y) d'entrainement")
ax.scatter(x_test, y_test, color='orange', marker='.', label=r"Données $(x,y) de test")
ax.scatter(x_test, y_test_predict, color='red', marker='.', label=fr"Prédictions de test $(x,\hat{{y}})$")
ax.plot([0,75],[intercept,intercept+coef*75],
        color='black',
        linestyle='--',
        label=rf"Modèle $y={intercept:2.2f}{coef:+2.2f}x$"
)

ax.set_xlabel("Biking to work (% of population)")
ax.set_ylabel("Heart disease (% of population)")

ax.legend(loc='best')

plt.show()

In [ ]:
print(f"MAE:  {metrics.mean_absolute_error(y_test, y_test_predict):2.3f}")
print(f"RMSE: {np.sqrt(metrics.mean_squared_error(y_test, y_test_predict)):2.3f}")